# DashBot Reward Logic Notebook

Notebook nay dung de kiem tra cac module nen tang truoc khi train A3C:

- DataProfiler: suy luan Q/N/T va thong ke cot
- InsightDetector: distribution, trend, correlation, top/bottom k
- RewardEngine: diversity, parsimony, insight reward
- DashboardEnv: reset/step voi action `add`
- GreedyDashboardRecommender: baseline de demo product khi A3C chua train xong

Chay notebook tu root project `DashBot/` hoac de cell setup tu them `backend/` vao `sys.path`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'backend').exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / 'backend'))
ROOT

In [ ]:
import pandas as pd

from dashbot.core.data_profiler import DataProfiler
from dashbot.core.insight_detector import InsightDetector
from dashbot.core.chart_generator import ChartGenerator
from dashbot.core.models import ChartSpec
from dashbot.core.recommender import GreedyDashboardRecommender
from dashbot.rl_env.rewards import RewardEngine
from dashbot.rl_env.dashboard_env import DashboardEnv
from dashbot.rl_env.constraints import ConstraintSampler

csv_path = ROOT / 'data' / 'raw' / 'cars_sample.csv'
df = pd.read_csv(csv_path)
df.head()

## 1. Data Profiling

Paper dung handcrafted column features theo huong VizML. Ban reproduction nay gom: type, missing ratio, cardinality, unique ratio, entropy, Gini, min/max/mean/std/skewness.

In [ ]:
profiler = DataProfiler(max_modeled_columns=10)
profile = profiler.profile(df)

profile_table = pd.DataFrame([c.to_dict() for c in profile.columns])
profile_table

## 2. Insight Detection

Thu mot so chart spec noi bo. Cac chart nay tuong ung voi parameter space cua paper: mark + x/y/color + aggregate/bin.

In [ ]:
charts = [
    ChartSpec('bar', x='origin', y='mpg', y_agg='mean', title='Average MPG by origin'),
    ChartSpec('point', x='horsepower', y='mpg', title='MPG vs Horsepower'),
    ChartSpec('line', x='year', y='mpg', y_agg='mean', title='MPG over year'),
    ChartSpec('bar', x='mpg', x_agg='bin', title='Distribution of MPG'),
]

detector = InsightDetector(correlation_threshold=0.5, top_k=5)
insights = detector.detect_dashboard(df, profile, charts)
pd.DataFrame([insight.to_dict() for insight in insights])

## 3. Reward Engine

Cong thuc theo paper:

`cr_i = 0.33 * diversity + 0.33 * parsimony + 0.1 * insight_reward`

`r_i = cr_i - cr_{i-1}`

In [ ]:
reward_engine = RewardEngine(alpha=3.0, n_best=4, n_max=8)

rows = []
for n in range(0, 9):
    partial = charts[:min(n, len(charts))]
    rows.append({
        'chart_count': n,
        'parsimony': reward_engine.parsimony(n),
        'chart_type_diversity': reward_engine.chart_type_diversity(partial),
        'column_diversity': reward_engine.column_diversity(partial, profile),
        'insight_reward': reward_engine.insight_reward(df, profile, partial),
        'dashboard_reward': reward_engine.dashboard_reward(df, profile, partial),
    })

pd.DataFrame(rows)

In [ ]:
previous = charts[:1]
current = charts[:2]

reward_engine.immediate_reward(df, profile, previous, current)

## 4. Constrained Sampling Masks

Paper nhan manh mask truoc softmax. Cell nay xem action/column/aggregate nao dang hop le trong mot state.

In [ ]:
sampler = ConstraintSampler(max_charts=8)
masks = sampler.masks(profile, charts=charts[:1], key_column='origin', selected_field='origin')

{
    'actions': masks.actions,
    'columns': masks.columns,
    'aggregates_for_origin': masks.aggregates,
    'valid_marks_origin_mpg': sampler.valid_marks_for_fields(profile, 'origin', 'mpg'),
    'valid_marks_horsepower_mpg': sampler.valid_marks_for_fields(profile, 'horsepower', 'mpg'),
}

## 5. DashboardEnv Step Demo

Day la cau noi research RL: agent chon action, env execute, reward engine tra diem delta.

In [ ]:
env = DashboardEnv(df)
state = env.reset(key_column='origin')
state.to_dict()

In [ ]:
chart = ChartSpec('bar', x='origin', y='mpg', y_agg='mean', title='Average MPG by origin')
next_state, reward, done, info = env.step('add', {'chart': chart})

{
    'reward_delta': reward,
    'done': done,
    'info': info,
    'state': next_state.to_dict(),
}

## 6. Greedy Baseline Recommendation

Baseline nay khong phai A3C. No dung reward engine de chon chart tham lam. Dung de demo product va sanity-check reward truoc khi train.

In [ ]:
recommender = GreedyDashboardRecommender(max_charts=5)
result = recommender.recommend(df)

print('key_column:', result['key_column'])
print('reward:', round(result['reward'], 4))
print('charts:', len(result['charts']))
print('insights:', len(result['insights']))

pd.DataFrame([{k: v for k, v in chart.items() if k != 'vega_lite'} for chart in result['charts']])

In [ ]:
pd.DataFrame(result['insights']).head(20)

## 7. Vega-Lite Spec Preview

Frontend hien tai dung Chart.js, nhung backend da co generator theo huong Vega-Lite nhu paper.

In [ ]:
generator = ChartGenerator()
generator.to_vega_lite(charts[0], profile=profile)

## 8. A3C Gap: Feature Tensor + Rollout Collector

Network da co trong `backend/dashbot/agent/networks.py`, nhung de train that su can them hai cau noi:

1. `DashboardState -> torch.Tensor`: ma hoa chart features, key column, all column features, padding.
2. Rollout loop: model sample action/parameters, goi `env.step`, luu `state/action/reward/log_prob/value/entropy`, roi update trainer.

Pseudo-code:

```python
state = env.reset()
for step in range(500_000):
    features = encode_state(state, env.profile)
    masks = build_torch_masks(state, env.profile)
    outputs = model(features, masks)
    action, params, log_prob, entropy = sample_action_and_params(outputs)
    next_state, reward, done, info = env.step(action, params)
    buffer.append(state, action, reward, done, log_prob, outputs['value'], entropy)
    state = env.reset() if done else next_state
    if len(buffer) >= rollout_length:
        trainer.update(buffer)
        buffer.clear()
```
